# Example 05: JSH-001-004-01

실제 mzML 데이터를 사용한 분석 예제입니다.

| Step | 내용 |
|------|------|
| 1 | 환경 설정 |
| 2 | mzML 로드 → TIC, 스펙트럼 가시화 |
| 3 | 피크 검출 & 적분 |
| 4 | 화합물 구조 입력 & 패턴 매칭 |
| 5 | AI 추론: Side product 탐색 |
| 6 | 3중 검증 |
| 7 | 최종 판정 |


## Step 1. 환경 설정

In [ ]:
!pip install -q pyopenms numpy scipy matplotlib seaborn pandas

!git clone -q https://github.com/chsong-hash/echo-tof-colab.git /content/etof 2>/dev/null || true
import sys; sys.path.insert(0, '/content/etof/src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyopenms as oms

plt.rcParams['figure.figsize'] = (14, 4)
sns.set_style("whitegrid")

from echo_tof.molecule import Molecule
from echo_tof.isotope_calc import IsotopicDistributionCalculator
from echo_tof.pattern import MoleculePattern
from echo_tof.filters import FormulaFilter
from echo_tof.calculations import get_mass_error
from echo_tof_ext.di_spectrum import pick_peaks, group_isotope_clusters, extract_cluster_at_mz
from echo_tof_ext.isotope_verifier import IsotopeVerifier
from echo_tof_ext.mz_predict import predict_mz
from echo_tof_ext.reaction_predictor import predict_byproduct_mws, validate_compound_by_adducts
from echo_tof_ext.neutral_loss_db import NEUTRAL_LOSSES

print("Ready!")


## Step 2. mzML 로드 & 가시화

파일: `JSH-001-004-01.mzML`

In [ ]:
# ─── mzML 로드 ───
# Colab: Google Drive에서 로드하거나 업로드
# from google.colab import files
# uploaded = files.upload()

MZML_PATH = "JSH-001-004-01.mzML"  # 실제 경로로 수정

exp = oms.MSExperiment()
oms.MzMLFile().load(MZML_PATH, exp)

ms1_spectra = [s for s in exp if s.getMSLevel() == 1]
print(f"File: {MZML_PATH}")
print(f"MS1 spectra: {len(ms1_spectra)}")
print(f"RT range: {ms1_spectra[0].getRT():.1f} - {ms1_spectra[-1].getRT():.1f} sec")


In [ ]:
# ─── TIC (Total Ion Chromatogram) ───
rts = [s.getRT() for s in ms1_spectra]
tics = [np.sum(s.get_peaks()[1]) for s in ms1_spectra]

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

ax = axes[0]
ax.plot(rts, tics, color='#0078D4', lw=0.5)
ax.set_xlabel('Retention Time (sec)'); ax.set_ylabel('TIC')
ax.set_title('Total Ion Chromatogram')

# ─── 대표 스펙트럼 (TIC 최대 지점) ───
max_idx = np.argmax(tics)
best_spec = ms1_spectra[max_idx]
mz, intensity = best_spec.get_peaks()

ax = axes[1]
ax.plot(mz, intensity, color='#E85D04', lw=0.5)
ax.set_xlabel('m/z'); ax.set_ylabel('Intensity')
ax.set_title(f'Spectrum at RT={rts[max_idx]:.1f}s (TIC max)')

plt.tight_layout(); plt.show()

print(f"Best spectrum: RT={rts[max_idx]:.1f}s, {len(mz)} points, max_int={intensity.max():.0f}")


In [ ]:
# ─── Custom Data 추출 (전체 스펙트럼 평균 or TIC max) ───
# 여기서는 TIC 최대 지점의 스펙트럼을 사용
custom_data = pd.DataFrame({'mz': mz, 'intensity': intensity})
print(f"Custom Data: {len(custom_data)} points")
custom_data.head(10)


## Step 3. 피크 검출 & 적분

In [ ]:
# ─── 피크 검출 (echo_tof_ext) ───
detected = pick_peaks(mz, intensity, noise_factor=3.0, min_intensity_pct=0.5)
peaks_df = pd.DataFrame(detected)
clusters = group_isotope_clusters(detected, charge=1, mz_tolerance=0.02)

print(f"Peaks: {len(peaks_df)}, Clusters: {len(clusters)}")

# ────────────────────────────────────────────
# [USER] VSCode 적분 로직을 여기에 삽입하세요:
# from my_module import custom_integrate
# peaks_df['custom_area'] = ...
# ────────────────────────────────────────────

# 적분 결과 가시화
fig, axes = plt.subplots(2, 1, figsize=(15, 7), gridspec_kw={'height_ratios': [3, 1]})
ax = axes[0]
ax.plot(mz, intensity, color='#888', lw=0.3, alpha=0.5)
for _, pk in peaks_df.iterrows():
    idx = int(pk['index'])
    l, r = max(0, idx-5), min(len(mz)-1, idx+5)
    ax.fill_between(mz[l:r+1], 0, intensity[l:r+1], alpha=0.3, color='#2196F3')
ax.set_ylabel('Intensity'); ax.set_title(f'Peak Integration: {len(peaks_df)} peaks')

ax = axes[1]
if not peaks_df.empty:
    ax.bar(peaks_df['mz'], peaks_df['area'], width=0.5, color='#0078D4', alpha=0.7)
ax.set_xlabel('m/z'); ax.set_ylabel('Area')
plt.tight_layout(); plt.show()

peaks_df.nlargest(15, 'area')[['mz','intensity','area','sn']].round(2)


## Step 4. 화합물 구조 입력 & 패턴 매칭

> 아래 `TARGET_FORMULA`와 `TARGET_MW`를 실제 타겟 화합물로 수정하세요.


In [ ]:
# ─── 타겟 화합물 설정 (수정 필요) ───
TARGET_FORMULA = "C20H25NO4"     # ← 실제 화합물 분자식
TARGET_MW = None                  # None이면 자동 계산
CHARGE = 1

# MW 자동 계산
mol = Molecule(TARGET_FORMULA, charge=0)
if TARGET_MW is None:
    TARGET_MW = mol.monoisotopic_mass
PROTON = 1.00728
target_mh = TARGET_MW + PROTON

# Adduct 테이블
adducts = predict_mz(TARGET_MW, mode="positive")
print(f"Target: {TARGET_FORMULA} (MW={TARGET_MW:.4f})")
print(f"Expected adducts:")
for a in adducts:
    print(f"  {a['adduct']:15s} m/z = {a['mz']:.4f}")

# 이론 동위원소 패턴
idc = IsotopicDistributionCalculator(ppm_tolerance=True, tolerance=50.0)
mp = MoleculePattern(charge=CHARGE)
mp.calculate_pattern(mol, idc)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
# 이론 패턴
axes[0].bar(mp.pattern_mz, mp.pattern_rel_intensities, width=0.08, color='#2196F3', alpha=0.8)
axes[0].set_xlabel('m/z'); axes[0].set_title(f'{TARGET_FORMULA} Theoretical Pattern')
# 실측 영역
m = (mz >= target_mh - 2) & (mz <= target_mh + 5)
axes[1].plot(mz[m], intensity[m], color='#E85D04', lw=0.8)
axes[1].set_xlabel('m/z'); axes[1].set_title(f'Experimental Region (m/z {target_mh:.1f})')
plt.tight_layout(); plt.show()


## Step 5. AI 추론: Side Product 탐색

In [ ]:
# ─── 반응 부산물 예측 ───
REACTION_TYPE = "amide_coupling"  # ← 실제 반응 유형
SM_MW = TARGET_MW                 # ← SM MW (다르면 수정)
REAGENT_MW = 120.0                # ← 시약 MW

byproducts = predict_byproduct_mws(SM_MW, REACTION_TYPE, REAGENT_MW)
print(f"Predicted byproducts: {len(byproducts)}")
for bp in byproducts:
    print(f"  {bp['name']:30s} MW={bp['mw']:.3f} ({bp['likelihood']})")

# ─── 중성 손실 후보 ───
nl_candidates = []
for nl_name, nl_mass, _ in NEUTRAL_LOSSES[:25]:
    exp_mz = target_mh - nl_mass
    if 50 < exp_mz < mz[-1]:
        nl_candidates.append({"name": f"-{nl_name}", "mz": exp_mz})
print(f"\nNeutral loss candidates: {len(nl_candidates)}")

# ─── 스펙트럼 탐색 ───
peak_mzs = np.array([p['mz'] for p in detected])
MZ_TOL = 0.03

found_sides = []
for bp in byproducts:
    bp_mh = bp['mw'] + PROTON
    matches = np.where(np.abs(peak_mzs - bp_mh) < MZ_TOL)[0]
    status = "FOUND" if len(matches) > 0 else "ABSENT"
    found_sides.append({"name": bp["name"], "mz": bp_mh, "detected": len(matches)>0, "type": "byproduct"})
    print(f"  [{status:6s}] {bp['name']:30s} m/z={bp_mh:.4f}")

for nl in nl_candidates[:10]:
    matches = np.where(np.abs(peak_mzs - nl['mz']) < MZ_TOL)[0]
    if len(matches) > 0:
        found_sides.append({"name": nl["name"], "mz": nl["mz"], "detected": True, "type": "neutral_loss"})
        print(f"  [FOUND ] {nl['name']:30s} m/z={nl['mz']:.4f}")

print(f"\nTotal side products found: {sum(1 for s in found_sides if s['detected'])}")


## Step 6. 3중 검증

In [ ]:
# ─── 6a. Mass Accuracy ───
matches = np.where(np.abs(peak_mzs - target_mh) < MZ_TOL)[0]
if len(matches) > 0:
    best = matches[np.argmax(np.array([detected[i]['intensity'] for i in matches]))]
    err_ppm = abs((peak_mzs[best] - target_mh) / target_mh * 1e6)
    mass_pass = err_ppm < 5.0
    print(f"6a Mass Accuracy: {err_ppm:.2f} ppm {'PASS' if mass_pass else 'FAIL'}")
else:
    err_ppm, mass_pass = 999, False
    print("6a Mass Accuracy: [M+H]+ NOT FOUND")

# ─── 6b. Isotope Pattern ───
verifier = IsotopeVerifier(ppm_tolerance=10.0)
cluster = extract_cluster_at_mz(mz, intensity, target_mh, 1, 10.0, 5)
if cluster["found"]:
    vr = verifier.verify(TARGET_FORMULA, CHARGE, cluster["mz_array"], cluster["int_array"])
    iso_pass = vr["grade"] in ("excellent", "good")
    print(f"6b Isotope: grade={vr['grade'].upper()}, CE={vr['cluster_error']:.4f} {'PASS' if iso_pass else 'FAIL'}")
else:
    iso_pass = False
    print("6b Isotope: cluster not extracted")

# ─── 6c. Adduct Consistency ───
peak_ints_arr = np.array([p['intensity'] for p in detected])
val = validate_compound_by_adducts(peak_mzs, peak_ints_arr, TARGET_MW, MZ_TOL, "positive")
adduct_pass = val["n_found"] >= 2
print(f"6c Adducts: {val['n_found']}/{val['n_possible']} {'PASS' if adduct_pass else 'FAIL'}")

# ─── Judgment ───
n_pass = sum([mass_pass, iso_pass, adduct_pass])
judgment = {3:"CONFIRMED", 2:"PROBABLE", 1:"TENTATIVE", 0:"NOT DETECTED"}[n_pass]
print(f"\n>>> JUDGMENT: {judgment} ({n_pass}/3)")


## Step 7. 최종 리포트

In [ ]:
# ─── 리포트 ───
print("=" * 60)
print(f"  Sample: JSH-001-004-01")
print("=" * 60)
print(f"  Target: {TARGET_FORMULA} (MW={TARGET_MW:.4f})")
print(f"  Peaks: {len(peaks_df)}, Clusters: {len(clusters)}")
print(f"  Mass Accuracy: {err_ppm:.2f} ppm ({'PASS' if mass_pass else 'FAIL'})")
print(f"  Isotope: {'PASS' if iso_pass else 'FAIL'}")
print(f"  Adducts: {val['n_found']}/{val['n_possible']} ({'PASS' if adduct_pass else 'FAIL'})")
print(f"  Side products: {sum(1 for s in found_sides if s['detected'])}")
print(f"  JUDGMENT: {judgment} ({n_pass}/3)")
print("=" * 60)

# 스펙트럼 + 판정 결과 가시화
JCOLOR = {"CONFIRMED":"#2E8B57", "PROBABLE":"#FF8C00", "TENTATIVE":"#DC143C", "NOT DETECTED":"#999"}
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(mz, intensity, color='#ccc', lw=0.5)

# Target compound
idx = np.argmin(np.abs(mz - target_mh))
color = JCOLOR[judgment]
ax.fill_between(mz[max(0,idx-8):idx+9], 0, intensity[max(0,idx-8):idx+9], color=color, alpha=0.5)
ax.annotate(f"{TARGET_FORMULA}\n{judgment}\nm/z={target_mh:.4f}",
    xy=(target_mh, intensity[idx]), xytext=(0, 20),
    textcoords='offset points', ha='center', fontsize=9,
    color=color, fontweight='bold',
    arrowprops=dict(arrowstyle='->', color=color))

# Side products
for s in found_sides:
    if s['detected']:
        si = np.argmin(np.abs(mz - s['mz']))
        sc = '#FF6B35' if s['type']=='byproduct' else '#9B59B6'
        ax.fill_between(mz[max(0,si-4):si+5], 0, intensity[max(0,si-4):si+5], color=sc, alpha=0.3)

ax.set_xlabel('m/z'); ax.set_ylabel('Intensity')
ax.set_title(f'JSH-001-004-01 — {judgment}')
plt.tight_layout(); plt.show()
